In [ ]:
import pandas as pd
import numpy as np
import re, os, json, joblib, warnings, time
from collections import Counter

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    VotingClassifier
)
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, r2_score,
    mean_squared_error, mean_absolute_error, silhouette_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.semi_supervised import LabelPropagation
import xgboost as xgb

warnings.filterwarnings('ignore')
np.random.seed(42)
# ── PATHS ──
DATA_DIR = '../data/processed'
SAVE_DIR = '../models/model_b/traditional'
os.makedirs(SAVE_DIR, exist_ok=True)
print("Setup complete!")

In [ ]:
start_time = time.time()

train_df = pd.read_csv(os.path.join(DATA_DIR, 'train_clean.csv'))
dev_df   = pd.read_csv(os.path.join(DATA_DIR, 'dev_clean.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test_clean.csv'))

for dset in [train_df, dev_df, test_df]:
    for col in ['A', 'B', 'C', 'D', 'A_clean', 'B_clean', 'C_clean', 'D_clean']:
        if col in dset.columns:
            dset[col] = dset[col].fillna('')

print(f"Train: {train_df.shape}")
print(f"Dev:   {dev_df.shape}")
print(f"Test:  {test_df.shape}")

STOPWORDS = frozenset('the a an is are was were in on at to for of and or but it he she they we i you that this with from by as not be has have had do does did will would can could may might shall should about up out if then than what which who whom when where why how all each every both few more most other some such no nor only own same so very just because its my his her their our your also been into'.split())

def clean_text(text):
    if not isinstance(text, str): return ""
    return re.sub(r'[^\\w\\s]', '', text.lower()).strip()

def tokenize(text):
    return clean_text(text).split()

def content_tokens(text):
    return set(tokenize(text)) - STOPWORDS

def jaccard(a, b):
    sa, sb = set(tokenize(a)), set(tokenize(b))
    u = sa | sb
    return len(sa & sb) / len(u) if u else 0.0

def overlap_ratio(a, b):
    ta, tb = set(tokenize(a)), set(tokenize(b))
    return len(ta & tb) / len(ta) if ta else 0.0

def sentence_split(article):
    sents = re.split(r'(?<=[.!?])\\s+', str(article))
    return [s.strip() for s in sents if len(s.strip()) > 10]

print("Data loaded & helpers ready!")

In [ ]:
print("=" * 70)
print("BUILDING DISTRACTOR RANKING FEATURES")
print("=" * 70)

def build_distractor_features(df, desc=""):
    rows = []
    for _, r in df.iterrows():
        article      = str(r['article'])
        question     = str(r['question'])
        ans_label    = r['answer']
        correct_text = str(r[ans_label])
        sents = sentence_split(article)
        correct_ct = content_tokens(correct_text)
        q_ct       = content_tokens(question)

        for opt in 'ABCD':
            opt_text  = str(r[opt])
            opt_ct    = content_tokens(opt_text)
            is_dist   = int(opt != ans_label)

            ohe_cos   = r.get(f'ohe_cos_art_{opt}', 0.0)
            tfidf_cos = r.get(f'tfidf_cos_art_{opt}', 0.0)
            jac_q     = r.get(f'jaccard_q_{opt}', 0.0)
            art_ov    = r.get(f'{opt}_article_overlap', 0.0)
            q_ov      = r.get(f'{opt}_question_overlap', 0.0)
            wc        = r.get(f'{opt}_word_count', len(tokenize(opt_text)))

            correct_wc   = len(tokenize(correct_text))
            len_ratio    = wc / max(correct_wc, 1)
            char_count   = len(opt_text)
            correct_cc   = len(correct_text)
            cc_ratio     = char_count / max(correct_cc, 1)
            correct_jac  = jaccard(opt_text, correct_text)
            correct_ov   = overlap_ratio(opt_text, correct_text)
            shared_content  = len(opt_ct & correct_ct) / max(len(opt_ct | correct_ct), 1)
            q_content_share = len(opt_ct & q_ct) / max(len(opt_ct | q_ct), 1)

            opt_clean = clean_text(opt_text)
            in_article = 1 if opt_clean in clean_text(article) else 0
            in_sentence = 1 if any(opt_clean in clean_text(s) for s in sents) else 0

            pos = article.lower().find(opt_clean[:min(20, len(opt_clean))])
            position = pos / max(len(article), 1) if pos >= 0 else -1.0

            corr_art_ov = r.get('correct_option_art_overlap', 0.0)
            art_ov_diff = abs(float(art_ov) - float(corr_art_ov))
            unique_to_opt = len(opt_ct - q_ct - correct_ct)

            rows.append({
                'is_distractor':    is_dist,
                'ohe_cos_art':      float(ohe_cos) if pd.notna(ohe_cos) else 0.0,
                'tfidf_cos_art':    float(tfidf_cos) if pd.notna(tfidf_cos) else 0.0,
                'jaccard_q':        float(jac_q) if pd.notna(jac_q) else 0.0,
                'art_overlap':      float(art_ov),
                'q_overlap':        float(q_ov),
                'word_count':       int(wc),
                'len_ratio':        len_ratio,
                'char_count':       char_count,
                'cc_ratio':         cc_ratio,
                'correct_jaccard':  correct_jac,
                'correct_overlap':  correct_ov,
                'shared_content':   shared_content,
                'q_content_share':  q_content_share,
                'in_article':       in_article,
                'in_sentence':      in_sentence,
                'position':         position,
                'art_ov_diff':      art_ov_diff,
                'unique_to_opt':    unique_to_opt,
            })

    result_df = pd.DataFrame(rows)
    feat_cols = [c for c in result_df.columns if c != 'is_distractor']
    X = np.nan_to_num(result_df[feat_cols].values.astype(float), nan=0.0)
    y = result_df['is_distractor'].values
    if desc:
        print(f"  {desc}: {X.shape[0]} samples, {X.shape[1]} features, "
              f"correct={sum(y==0)}, distractor={sum(y==1)}")
    return X, y, feat_cols

X_train, y_train, feat_cols = build_distractor_features(train_df, "Train")
X_dev,   y_dev,   _         = build_distractor_features(dev_df,   "Dev")
X_test,  y_test,  _         = build_distractor_features(test_df,  "Test")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_dev_sc   = scaler.transform(X_dev)
X_test_sc  = scaler.transform(X_test)

print("Feature extraction complete!")

In [ ]:
print("=" * 70)
print("SUPERVISED DISTRACTOR RANKER MODELS")
print("=" * 70)

results_b1 = {}
predictions_dev = {}
predictions_test = {}

def train_eval(name, model, X_tr, y_tr, X_val, y_val, X_te, y_te):
    model.fit(X_tr, y_tr)
    p_val = model.predict(X_val)
    p_te  = model.predict(X_te)

    results_b1[name] = {
        'dev_accuracy':  accuracy_score(y_val, p_val),
        'dev_f1':        f1_score(y_val, p_val, average='macro'),
        'dev_precision': precision_score(y_val, p_val, average='macro'),
        'dev_recall':    recall_score(y_val, p_val, average='macro'),
        'test_accuracy': accuracy_score(y_te, p_te),
        'test_f1':       f1_score(y_te, p_te, average='macro'),
        'test_precision': precision_score(y_te, p_te, average='macro'),
        'test_recall':   recall_score(y_te, p_te, average='macro'),
    }
    predictions_dev[name] = p_val
    predictions_test[name] = p_te

    print(f"\\n{name}")
    print(f"  Dev  — Acc: {results_b1[name]['dev_accuracy']:.4f}  F1: {results_b1[name]['dev_f1']:.4f}")
    print(f"  Test — Acc: {results_b1[name]['test_accuracy']:.4f}  F1: {results_b1[name]['test_f1']:.4f}")
    print(classification_report(y_val, p_val, target_names=['Correct', 'Distractor']))
    return model

lr = train_eval('Logistic Regression', LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced', random_state=42), X_train_sc, y_train, X_dev_sc, y_dev, X_test_sc, y_test)
svm = train_eval('SVM (LinearSVC)', LinearSVC(max_iter=3000, C=1.0, class_weight='balanced', random_state=42), X_train_sc, y_train, X_dev_sc, y_dev, X_test_sc, y_test)
rf = train_eval('Random Forest', RandomForestClassifier(n_estimators=200, max_depth=15, class_weight='balanced', random_state=42, n_jobs=-1), X_train, y_train, X_dev, y_dev, X_test, y_test)
xgb_m = train_eval('XGBoost', xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss', n_jobs=-1), X_train, y_train, X_dev, y_dev, X_test, y_test)
nb = train_eval('Naive Bayes', GaussianNB(), X_train_sc, y_train, X_dev_sc, y_dev, X_test_sc, y_test)

In [ ]:
print("=" * 70)
print("ENSEMBLE — SOFT VOTING")
print("=" * 70)

ens = VotingClassifier(estimators=[
    ('lr', LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced', random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=200, max_depth=15, class_weight='balanced', random_state=42, n_jobs=-1)),
    ('xgb', xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss', n_jobs=-1)),
], voting='soft')

ens = train_eval('Ensemble (Soft Vote)', ens, X_train_sc, y_train, X_dev_sc, y_dev, X_test_sc, y_test)

In [ ]:
print("=" * 70)
print("HYPERPARAMETER TUNING (XGBoost)")
print("=" * 70)

param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators': [200, 300, 400],
}
gs = GridSearchCV(
    xgb.XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1),
    param_grid, cv=3, scoring='f1_macro', n_jobs=-1, verbose=1
)
gs.fit(X_train, y_train)

best_xgb = gs.best_estimator_
p_dev  = best_xgb.predict(X_dev)
p_test = best_xgb.predict(X_test)
predictions_dev['XGBoost (Tuned)'] = p_dev
predictions_test['XGBoost (Tuned)'] = p_test
results_b1['XGBoost (Tuned)'] = {
    'dev_accuracy':  accuracy_score(y_dev, p_dev),
    'dev_f1':        f1_score(y_dev, p_dev, average='macro'),
    'dev_precision': precision_score(y_dev, p_dev, average='macro'),
    'dev_recall':    recall_score(y_dev, p_dev, average='macro'),
    'test_accuracy': accuracy_score(y_test, p_test),
    'test_f1':       f1_score(y_test, p_test, average='macro'),
    'test_precision': precision_score(y_test, p_test, average='macro'),
    'test_recall':   recall_score(y_test, p_test, average='macro'),
}

print(f"\\nBest params: {gs.best_params_}")
print(f"Best CV F1: {gs.best_score_:.4f}")
print(f"Dev  — Acc: {results_b1['XGBoost (Tuned)']['dev_accuracy']:.4f}  F1: {results_b1['XGBoost (Tuned)']['dev_f1']:.4f}")
print(f"Test — Acc: {results_b1['XGBoost (Tuned)']['test_accuracy']:.4f}  F1: {results_b1['XGBoost (Tuned)']['test_f1']:.4f}")

In [ ]:
print("=" * 70)
print("MODEL B2 — HINT SENTENCE SCORER")
print("=" * 70)

def build_hint_features(df, desc=""):
    rows = []
    labels = []
    for _, r in df.iterrows():
        article  = str(r['article'])
        question = str(r['question'])
        correct  = str(r[r['answer']])
        sents    = sentence_split(article)
        if not sents:
            continue
        key_ct = content_tokens(question) | content_tokens(correct)

        for si, sent in enumerate(sents):
            s_ct = content_tokens(sent)
            relevance = len(s_ct & key_ct) / max(len(key_ct), 1)
            rows.append({
                'position':        si / max(len(sents)-1, 1),
                'sent_word_count': len(tokenize(sent)),
                'q_jaccard':       jaccard(sent, question),
                'ans_jaccard':     jaccard(sent, correct),
                'q_overlap':       overlap_ratio(question, sent),
                'ans_overlap':     overlap_ratio(correct, sent),
                'keyword_overlap': relevance,
                'ans_in_sent':     int(clean_text(correct) in clean_text(sent)),
                'content_density': len(s_ct) / max(len(tokenize(sent)), 1),
            })
            labels.append(relevance)

    result_df = pd.DataFrame(rows)
    X = np.nan_to_num(result_df.values.astype(float), nan=0.0)
    y = np.array(labels)
    if desc:
        print(f"  {desc}: {X.shape[0]} sentences, {X.shape[1]} features")
    return X, y

train_hint_sample = train_df.sample(n=min(10000, len(train_df)), random_state=42)
Xh_train, yh_train = build_hint_features(train_hint_sample, "Train (sampled)")
Xh_dev,   yh_dev   = build_hint_features(dev_df, "Dev")

thr = np.percentile(yh_train, 80)
yh_train_bin = (yh_train >= thr).astype(int)
yh_dev_bin   = (yh_dev >= thr).astype(int)

hint_lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
hint_lr.fit(Xh_train, yh_train_bin)
hp_dev = hint_lr.predict(Xh_dev)

hint_rf = RandomForestRegressor(n_estimators=150, max_depth=10, random_state=42, n_jobs=-1)
hint_rf.fit(Xh_train, yh_train)

print("Hint models trained!")

In [ ]:
def get_option_candidates_from_dataset(df, n=5000):
    pool = {}
    for _, r in df.sample(n=min(n, len(df)), random_state=42).iterrows():
        for opt in 'ABCD':
            text = str(r[opt]).strip()
            if text:
                wc = len(tokenize(text))
                if wc not in pool:
                    pool[wc] = []
                pool[wc].append(text)
    return pool

print("Building option pool...")
option_pool = get_option_candidates_from_dataset(train_df, n=5000)
joblib.dump(option_pool, os.path.join(SAVE_DIR, 'option_pool.pkl'))
print(f"Option pool saved to {os.path.join(SAVE_DIR, 'option_pool.pkl')}")

In [ ]:
print("=" * 70)
print("SAVING MODELS")
print("=" * 70)

saves = {
    'distractor_lr.pkl': lr, 'distractor_svm.pkl': svm,
    'distractor_rf.pkl': rf, 'distractor_xgb.pkl': xgb_m,
    'distractor_xgb_tuned.pkl': best_xgb, 'distractor_nb.pkl': nb,
    'distractor_ensemble.pkl': ens, 'scaler.pkl': scaler,
    'hint_classifier_lr.pkl': hint_lr, 'hint_regressor_rf.pkl': hint_rf,
    'feature_columns.json': feat_cols,
}
for fname, obj in saves.items():
    path = os.path.join(SAVE_DIR, fname)
    if fname.endswith('.json'):
        with open(path, 'w') as f: json.dump(obj, f)
    else:
        joblib.dump(obj, path)
    print(f"  Saved: {path}")

elapsed = time.time() - start_time
print(f"\\nTotal time: {elapsed:.1f}s")
print("=" * 70)
print("MODEL B TRAINING COMPLETE")
print("=" * 70)